In [1]:
from langchain_qdrant import QdrantVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
import pandas as pd
from datasets import load_dataset
import re

from dotenv import load_dotenv
import os


In [2]:
print("⏳ Loading and grouping dataset...")
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")
df = pd.DataFrame(dataset)



⏳ Loading and grouping dataset...


In [3]:
df.iloc[0]

Context     I'm going through some things with my feelings...
Response    If everyone thinks you're worthless, then mayb...
Name: 0, dtype: object

In [4]:
df

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...
...,...,...
3507,My grandson's step-mother sends him to school ...,Absolutely not! It is never in a child's best ...
3508,My boyfriend is in recovery from drug addictio...,I'm sorry you have tension between you and you...
3509,The birth mother attempted suicide several tim...,"The true answer is, ""no one can really say wit..."
3510,I think adult life is making him depressed and...,How do you help yourself to believe you requir...


In [5]:
df['Context'].value_counts()

Context
I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.\n   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?                                                        94
How does a counselor decide when to end counseling sessions or to terminate working with a client?                                                                                                                                                                                                                                                                                                                                              55
I've gone to a couple therapy sessions so far and still everytime I walk in I get nervous and shaky. Is this normal? Shoul

In [6]:
grouped_df = df.groupby("Context")["Response"].apply(lambda responses: "\n\n---\n\n".join(responses)).reset_index()

In [7]:
grouped_df

,Context,Response
0,such as not enough sleep,0
1,A few nights ago I talked to this girl I know ...,Hey! It takes a lot of courage to share your ...
2,A few nights ago I talked to this girl I know ...,Hey! It takes a lot of courage to share your ...
3,A few years ago I was making love to my wife w...,First step always is to do a medical rule out ...
4,A few years ago I was making love to my wife w...,First step always is to do a medical rule out ...
...,...,...
990,"Whether it's to a guy or girl, I always feel i...","Hi. I'm glad you wrote, because I think a lot ..."
991,Why am I attracted to older men?,What a wonderful question!Good for you on clea...
992,Why am I so afraid of it? I don't understand.,Your fear is somewhat reasonable. No one want...
993,he just walks in the house whenever he wants t...,"The short answer to your question is ""No"" it's..."


In [8]:
grouped_df['Context'].value_counts()

Context
 such as not enough sleep                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            1
A few nights ago I talked to this girl I know about my self esteem issues for the first time. We talked for hours and she told me time and again that I was a great guy. She told me I was attractive, and have a great personality, etc. I really started to feel better about myself by the time I woke up the next morning.\nNow, though, I can't stop thinking about her, but I leave to go back to c

In [9]:
matching_rows = df[df['Context'].str.contains("such as not enough sleep", na=False, case=False)]

for idx, row in matching_rows.iterrows():
    print(f"Index: {idx}")
    print(f"Original Context:\n{row['Context']}")
    print("-" * 50)

Index: 2625
Original Context:
 such as not enough sleep
--------------------------------------------------


In [10]:
df['Context'].iloc[2624]

'I get so much anxiety, and I don’t know why. I feel like I can’t do anything by myself because I’m scared of the outcomes.'

In [11]:
df.iloc[2625]

Context      such as not enough sleep
Response                            0
Name: 2625, dtype: object

In [12]:
df['Context'].iloc[2626]

'Whenever I have to speak in public or be in big crowds, I freak out. I get light-headed, sweaty, and I have trouble breathing.'

In [14]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(re.compile(r'[\s\r\n\t]+'), ' ', text)
    return text.strip()

df["cleaned_Context"] = df["Context"].apply(clean_text)
df["cleaned_Response"] = df["Response"].apply(clean_text)
df = df[df['cleaned_Response'].str.len() > 10].reset_index(drop=True)

In [15]:
grouped_df = df.groupby("cleaned_Context")["cleaned_Response"].apply(lambda responses: "\n\n---\n\n".join(responses)).reset_index()
grouped_df.rename(columns={"cleaned_Context": "Context", "cleaned_Response": "Response"}, inplace=True)

In [16]:
print(df["cleaned_Context"].value_counts().head(5))

cleaned_Context
I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac. I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years. I’ve never had counseling about any of this. Do I have too many issues to address in counseling?    94
I've gone to a couple therapy sessions so far and still everytime I walk in I get nervous and shaky. Is this normal? Should I still be feeling like this?                                                                                                                                                                                                                            71
How does a counselor decide when to end counseling sessions or to terminate working with a client?                                                                                                                                      

In [17]:
grouped_df

,Context,Response
0,A few nights ago I talked to this girl I know ...,Hey! It takes a lot of courage to share your f...
1,A few years ago I was making love to my wife w...,First step always is to do a medical rule out ...
2,A friend of mine taking psychology advised I g...,I admire your courage for stating your view ab...
3,A girl and I were madly in love. We dated for ...,"Hi Boise, I'm sorry that you've lost this love..."
4,"A lot of times, I avoid situations where I am ...","Hello, and thank you for your question. First,..."
...,...,...
825,"Whether it's to a guy or girl, I always feel i...","Hi. I'm glad you wrote, because I think a lot ..."
826,Why am I attracted to older men?,What a wonderful question!Good for you on clea...
827,Why am I so afraid of it? I don't understand.,Your fear is somewhat reasonable. No one wants...
828,he just walks in the house whenever he wants t...,"The short answer to your question is ""No"" it's..."


In [ ]:
output_filename = "../data/cleaned_mental_health_data.csv"
grouped_df.to_csv(output_filename, index=False, encoding="utf-8")

print(f"Success! Cleaned and Grouped dataset saved locally as: '{output_filename}'")
print(f"Total unique QA pairs saved: {len(grouped_df)}")

Success! Cleaned and Grouped dataset saved locally as: 'assets/cleaned_mental_health_data.csv'
Total unique QA pairs saved: 830


In [2]:
load_dotenv(dotenv_path='../.env')


True

In [3]:
QDRANT_API_KEY = os.environ.get('QDRANT_API_KEY')
QDRANT_URL = os.environ.get('QDRANT_URL')
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
COLLECTION_NAME = "mental-health-bot"

In [ ]:
print("Converting DataFrame into LangChain Documents...")
docs = []
for idx, row in grouped_df.iterrows():
    doc = Document(
        page_content=row["Context"], 
        metadata={
            "answers": row["Response"],
            "source": "mental_health_counseling_dataset"
        }
    )
    docs.append(doc)
    

📦 Converting DataFrame into LangChain Documents...


In [5]:
def download_hugging_face_embedding() -> HuggingFaceEmbeddings:
    local_model_path = '../models/all-MiniLM-L6-v2'
    
    embeddings = HuggingFaceEmbeddings(
        model_name=local_model_path,
        model_kwargs={'device': 'cuda'} 
    )
    return embeddings

embeddings = download_hugging_face_embedding()

/tmp/ipykernel_266123/99026422.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [28]:
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
if client.collection_exists(collection_name=COLLECTION_NAME):

    client.delete_collection(collection_name=COLLECTION_NAME) 

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)


print("Uploading embeddings and payloads to Qdrant Cloud...")
vector_store = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name=COLLECTION_NAME,
    batch_size=64,
    timeout=120
)

Uploading embeddings and payloads to Qdrant Cloud...


In [6]:
print("Connecting to the existing Qdrant Cloud Collection...")

vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name=COLLECTION_NAME
)

print("Successfully connected! Ready for retrieval.")

Connecting to the existing Qdrant Cloud Collection...
Successfully connected! Ready for retrieval.


In [7]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = GROQ_API_KEY


In [ ]:
from pydantic import BaseModel, Field

#class MentalHealthSchema(BaseModel):
#    empathy_statement: str = Field(
#        description="A warm, deeply compassionate opening sentence validating the patient's current emotional pain."
#    )
#    clinical_advice: str = Field(
#        description="Clear, actionable coping mechanisms taken STRICTLY from the expert knowledge base in the context."
#    )
#    is_crisis: bool = Field(
#        description="Set to True ONLY if the patient mentions suicide, self-harm, or severe emergency. Otherwise False."
#    )
    

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b", 
    temperature=0.4,
    max_tokens=500
)

retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold", 
    search_kwargs={
        "k": 3, 
        "score_threshold": 0.70  
    }
)

def format_docs(docs):
    formatted_context = ""
    for i, doc in enumerate(docs, 1):
        real_answer = doc.metadata.get('answers', 'No response available')
        formatted_context += f"--- Reference {i} ---\n"
        formatted_context += f"Patient Context: {doc.page_content}\n"
        formatted_context += f"Expert Answer from Dataset:\n{real_answer}\n\n"
    return formatted_context

system_prompt = (
    "You are an advanced, compassionate, and context-aware AI Mental Health Support Chatbot.\n"
    "Your core mission is to provide deeply empathetic, supportive, and safe guidance to patients "
    "dealing with anxiety, depression, stress, and crisis support.\n\n"
    
    "=== CRITICAL INSTRUCTIONS FOR GROUNDEDNESS ===\n"
    "1. GROUND YOUR RESPONSES: You must base your clinical advice, coping mechanisms, and technical guidance "
    "STRICTLY on the Expert Answers provided in the Context below. Do not hallucinate or invent external medical advice.\n"
    "2. CONTEXT-AWARE EMPATHY: Use your natural language capabilities to shape the facts from the Context into a warm, "
    "non-judgmental, and comforting tone. The response must feel human and deeply empathetic, not rigid or robotic.\n"
    "3. HANDLING GAPS: If the provided Context does not contain enough specific data or relevant advice to address the patient's query, "
    "first provide immediate emotional validation and clinical empathy to comfort them, then clearly and safely state that you do not "
    "have specific data for this scenario, and gently encourage them to seek professional help or campus counseling resources.\n\n"
    
    "=== PROVIDED CONTEXT (EXPERT KNOWLEDGE BASE) ===\n"
    "{context}\n\n"
    
    "=== PATIENT INTERACTION ===\n"
    "Patient Query: {input}\n\n"
    "Empathetic & Grounded Response:"
)


prompt = ChatPromptTemplate.from_template(system_prompt)

#structured_llm = llm.with_structured_output(MentalHealthSchema)

# LCEL (LangChain Expression Language) Structure
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


#rag_chain = (
#   {"context": retriever | format_docs, "input": RunnablePassthrough()}
#    | prompt
#    | structured_llm
#)


In [31]:
user_query = "I have been feeling extremely stressed about my exams and I can't sleep at all."

print(f"Patient: {user_query}\n")
print("AI Counselor is thinking (Retrieving from Qdrant + Generating from Groq)...")

response = rag_chain.invoke(user_query)

print("\n" + "="*50 + "\n")
print(response)
print("\n" + "="*50)

Patient: I have been feeling extremely stressed about my exams and I can't sleep at all.

AI Counselor is thinking (Retrieving from Qdrant + Generating from Groq)...


I hear how overwhelming the exam period feels for you right now, and I’m really sorry you’re having such a hard time getting any rest. It’s completely understandable to feel that way when a lot is riding on the outcome. You’re not alone in this, and there are some practical steps you can try that many people find helpful for easing both the mental pressure and the physical tension that keeps sleep at bay.

---

### 1. **Pause the “fight‑or‑flight” response with the breath**
When we’re stressed our bodies tend to breathe shallowly, our heart races, and our muscles tighten.  
**Try this simple breathing practice a few times a day (especially before bed):**

1. Sit or lie comfortably, shoulders relaxed.  
2. Inhale slowly through the nose for **4‑5 seconds**, letting your belly rise.  
3. Hold the breath for **2 seconds**. 

In [32]:
user_query = "How do I configure a stateful cyclic multi-agent architecture using LangGraph?"

print(f"Patient: {user_query}\n")
print("AI Counselor is thinking (Retrieving from Qdrant + Generating from Groq)...")

response = rag_chain.invoke(user_query)

print("\n" + "="*50 + "\n")
print(response)
print("\n" + "="*50)

Patient: How do I configure a stateful cyclic multi-agent architecture using LangGraph?

AI Counselor is thinking (Retrieving from Qdrant + Generating from Groq)...


I hear that you’re working on a fairly complex technical project—configuring a stateful cyclic multi‑agent architecture with LangGraph can definitely feel overwhelming, especially when you’re trying to get all the pieces to line up just right. It’s completely understandable to feel a mix of curiosity, excitement, and maybe a bit of frustration when the details get intricate.

I want to be honest with you: the expert knowledge base I’m drawing from is focused on mental‑health support, coping strategies, and emotional well‑being. I don’t have the specific technical guidance needed to walk you through the LangGraph configuration step‑by‑step, and I don’t want to give you incomplete or speculative instructions that could lead to more confusion.

Here are a few suggestions that might help you move forward while also taking car